## Buyer Search Pipeline - Integration Test

This notebook verifies the buyer search pipeline end-to-end with live MongoDB data and an in-memory query fixture from `process_query()`. Run the startup check first so package and Python-version problems appear as readable output before the heavier cells run.


In [1]:
# ============================================
# STARTUP CHECK - Run this cell first
# ============================================
import sys
import subprocess

print(f"Python version: {sys.version}")
print(f"Platform: {sys.platform}")

# Check critical packages in the current kernel.
checks = {}
for pkg in ["pymongo", "pydantic", "numpy", "dotenv"]:
    try:
        __import__(pkg.replace("-", "_"))
        checks[pkg] = "\u2705 OK"
    except ImportError:
        checks[pkg] = "\u274c Missing"
    except Exception as e:
        checks[pkg] = f"\u26a0\ufe0f Error: {e}"

# Check heavy packages in a child process so a native crash does not kill this kernel.
for pkg in ["torch", "sentence_transformers"]:
    try:
        result = subprocess.run(
            [sys.executable, "-c", f"import {pkg}; print('ok')"],
            capture_output=True,
            text=True,
            timeout=45,
        )
        if result.returncode == 0:
            checks[pkg] = "\u2705 OK"
        else:
            error = (result.stderr or result.stdout or "import failed").strip().splitlines()[-1]
            checks[pkg] = f"\u26a0\ufe0f Error: {error}"
    except subprocess.TimeoutExpired:
        checks[pkg] = "\u26a0\ufe0f Error: import timed out"
    except Exception as e:
        checks[pkg] = f"\u26a0\ufe0f Error: {e}"

print("\nPackage check:")
for pkg, status in checks.items():
    print(f"  {pkg}: {status}")

# Check .env
from pathlib import Path
env_exists = (Path.cwd() / ".env").exists() or \
             (Path.cwd().parent / ".env").exists()
print(f"\n.env file: {'\u2705 Found' if env_exists else '\u274c Missing'}")


Python version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
Platform: win32

Package check:
  pymongo: ✅ OK
  pydantic: ✅ OK
  numpy: ✅ OK
  dotenv: ✅ OK
  torch: ✅ OK
  sentence_transformers: ✅ OK

.env file: ✅ Found


## Cell A - Safe Imports

This cell imports only lightweight dependencies and fixes `sys.path`. It matters because notebook working directories often differ between VSCode and Jupyter. Expected output is the detected project root.


In [2]:
# Cell A - Safe imports only
import json
import os
import sys
from pathlib import Path

import pymongo

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    get_ipython().run_line_magic("cd", str(ROOT.parent))
    ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Safe imports loaded.")
print("Project root:", ROOT)


c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer
Safe imports loaded.
Project root: c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer


## Cell B - MongoDB Connection

This cell loads `src.mongodb` inside a guarded import and checks Atlas connectivity. It matters because the search pipeline depends on indexed `items` and `retrieval_units`. Expected output is connection PASS and non-zero collection counts.


In [3]:
# Cell B - MongoDB imports and connection check
try:
    from src.mongodb import get_items_collection, get_retrieval_units_collection, ping_mongodb
    MONGODB_AVAILABLE = True
except ImportError as e:
    MONGODB_AVAILABLE = False
    print(f"WARNING: MongoDB helpers not available: {e}")
except Exception as e:
    MONGODB_AVAILABLE = False
    print(f"WARNING: Unexpected error loading MongoDB helpers: {e}")

if not MONGODB_AVAILABLE:
    raise RuntimeError("src.mongodb is required. Check imports above.")

items = get_items_collection()
retrieval_units = get_retrieval_units_collection()

ping = ping_mongodb()
print("MongoDB connection:", "PASS" if ping.get("ok") else "FAIL")
if not ping.get("ok"):
    raise RuntimeError(ping.get("error", "MongoDB ping failed"))

collection_counts = {
    "total_items": items.count_documents({}),
    "total_retrieval_units": retrieval_units.count_documents({}),
    "hype_question_units_with_embedding": retrieval_units.count_documents({
        "unit_type": "hype_question",
        "embedding": {"$exists": True},
    }),
    "proposition_units_with_text_search": retrieval_units.count_documents({
        "unit_type": "proposition",
        "text_search": {"$exists": True},
    }),
    "cold_start_items": items.count_documents({"cold_start.is_cold_item": True}),
}

print("\nCollection health")
print("Metric | Count")
print("--- | ---:")
for metric, count in collection_counts.items():
    print(f"{metric} | {count}")

collection_health_pass = all(count > 0 for count in collection_counts.values())
print("\nCollection health status:", "PASS" if collection_health_pass else "FAIL")


MongoDB connection: PASS

Collection health
Metric | Count
--- | ---:
total_items | 3000
total_retrieval_units | 29753
hype_question_units_with_embedding | 13580
proposition_units_with_text_search | 16173
cold_start_items | 3000

Collection health status: PASS


## Cell C - Search Pipeline Imports

This cell imports the MongoDB search pipeline separately from embedding code. It should not load torch or CUDA. Expected output is a simple confirmation that search imports loaded.


In [4]:
# Cell C - Search pipeline imports, no torch dependency expected
try:
    from src.search_pipeline import run_search
    from src.retrieval_output import build_explainable_result
    SEARCH_PIPELINE_AVAILABLE = True
except ImportError as e:
    SEARCH_PIPELINE_AVAILABLE = False
    print(f"WARNING: search_pipeline not available: {e}")
except Exception as e:
    SEARCH_PIPELINE_AVAILABLE = False
    print(f"WARNING: Unexpected error loading search_pipeline: {e}")

if SEARCH_PIPELINE_AVAILABLE:
    print("Search pipeline imports loaded.")


Search pipeline imports loaded.


## Cell D - Query Processor Boundary

This cell imports `query_processor` with a clear warning about BGE-M3. The actual model load happens when `process_query()` calls embedding. Expected output is either a clean import confirmation or a readable error message.


In [5]:
# Cell D - Heavy query processor import boundary
print("WARNING: Loading BGE-M3 model (~1.5GB) may take a few minutes when process_query() runs.")
print("Importing query_processor should be lightweight; torch/sentence-transformers load during embedding.")

try:
    from src.query_processor import process_query
    QUERY_PROCESSOR_AVAILABLE = True
except ImportError as e:
    QUERY_PROCESSOR_AVAILABLE = False
    print(f"WARNING: query_processor not available: {e}")
except Exception as e:
    QUERY_PROCESSOR_AVAILABLE = False
    print(f"WARNING: Unexpected error loading query_processor: {e}")

if QUERY_PROCESSOR_AVAILABLE:
    print("query_processor import loaded. Embedding model will load on first process_query() call.")


Importing query_processor should be lightweight; torch/sentence-transformers load during embedding.
query_processor import loaded. Embedding model will load on first process_query() call.


## Cell 3 - Build Test Fixture from Query Processor

This cell calls `process_query(TEST_QUERY)` to build the search fixture in memory. It matters because this is the real buyer-query contract: HyPE query, BM25 query, hard filters, and embedding. Expected output is a fixture preview with the embedding shortened for display.


In [6]:
# Cell 3 - Build Test Fixture from Query Processor
if "QUERY_PROCESSOR_AVAILABLE" not in dir() or not QUERY_PROCESSOR_AVAILABLE:
    raise RuntimeError("query_processor is required. Check imports above.")

TEST_QUERY = "Fast charge samsung galaxy a14 5g"
try:
    fixture = process_query(TEST_QUERY)
except Exception as e:
    raise RuntimeError(f"process_query failed. Check BGE-M3/torch/Ollama dependencies: {e}") from e

fixture_preview = dict(fixture)
embedding = fixture_preview.get("query_embedding")
if isinstance(embedding, list):
    fixture_preview["query_embedding"] = embedding[:3] + ["..."]

print(f"Built fixture from TEST_QUERY: {TEST_QUERY}")
print(json.dumps(fixture_preview, indent=2, ensure_ascii=False))


c:\HCMUS\MONGODB\coldstart+project\ColdStart_Killer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  7.15it/s]

Built fixture from TEST_QUERY: Fast charge samsung galaxy a14 5g
{
  "original_query": "Fast charge samsung galaxy a14 5g",
  "language_detected": "en",
  "english_query": "Fast charge samsung galaxy a14 5g",
  "hype_search_query_en": "user looking for Fast charge samsung galaxy a14 5g for everyday use",
  "bm25_search_query_en": "fast charge samsung galaxy a14 5g",
  "hard_filters": {
    "in_stock": true
  },
  "query_embedding": [
    0.02102605812251568,
    0.005477383732795715,
    -0.06972169876098633,
    "..."
  ]
}


## Cell 4 - Run Search Pipeline

This cell calls `run_search()` with mode `unionWith`. It verifies vector search, BM25 search, RRF fusion, item lookup, and explainable output formatting. Expected output is a raw result count greater than zero when Atlas indexes are ready.


In [7]:
# Cell 4 - Run Search Pipeline
if "fixture" not in dir() or fixture is None:
    raise RuntimeError("fixture chưa được tạo. Hãy chạy cell Query Processor trước.")
if "SEARCH_PIPELINE_AVAILABLE" not in dir() or not SEARCH_PIPELINE_AVAILABLE:
    raise RuntimeError("search_pipeline is required. Check imports above.")

results = run_search(fixture, mode="unionWith", top_k=10)
pipeline_returns_results_pass = len(results) > 0

print("Search mode: unionWith")
print("Top K: 10")
print("Raw result count:", len(results))


Search mode: unionWith
Top K: 10
Raw result count: 10


## Cell 5 - Results Table

This cell formats top results into a readable table. It helps verify score, vector rank, BM25 rank, matched channels, and cold-start notes. Expected output is a ranked table with short titles and channel information.


In [8]:
# Cell 5 — Results Table
if "results" not in dir():
    raise RuntimeError("results chưa có. Hãy chạy cell Search Execution trước.")

def preview_text(value, max_len=60):
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def result_channels(result):
    channels = (result.get("debug") or {}).get("matched_channels") or []
    return channels if isinstance(channels, list) else [channels]

print("Rank | Item ID | Title (60 chars) | Score | Vector Rank | BM25 Rank | Channels | Cold Start")
print("---: | --- | --- | ---: | ---: | ---: | --- | ---")
for rank, result in enumerate(results, start=1):
    channels = result_channels(result)
    print(
        f"{rank} | "
        f"{result.get('item_id')} | "
        f"{preview_text(result.get('title'))} | "
        f"{result.get('score')} | "
        f"{result.get('rank_vector')} | "
        f"{result.get('rank_bm25')} | "
        f"{', '.join(str(channel) for channel in channels)} | "
        f"{'yes' if result.get('cold_start_note') else 'no'}"
    )

hybrid_channels_pass = any(
    "vector" in result_channels(result) and "bm25" in result_channels(result)
    for result in results
)


Rank | Item ID | Title (60 chars) | Score | Vector Rank | BM25 Rank | Channels | Cold Start
---: | --- | --- | ---: | ---: | ---: | --- | ---
1 | B0BNK971SB | USB C Samsung Fast Charging Block Plug for Samsung Galaxy... | 0.11618527192297685 | 1 | 3 | bm25, vector | yes
2 | B08XWXYJFN | ZHIYIWU Designed for Samsung Galaxy A14 5G Case Clear Sho... | 0.11446527472527473 | 10 | 5 | vector, bm25 | yes
3 | B0BJB3JHB7 | USB Type C Wall Charger for iPhone 14, 20w Pd + Qc 3.0 Du... | 0.059375 | 4 | None | vector | yes
4 | B09KX9FQ4G | USB C Wall Plug, 2Pack 2-Port 20W Fast Wall Charger PD 3.... | 0.05895522388059701 | 7 | None | vector | yes
5 | B0BNHVDNTZ | QIVSTAR Galaxy A14 5G Wallet Case Embossed PU Leather Fli... | 0.05655737704918033 | None | 1 | bm25 | yes
6 | B0BHL4LVRM | Samsung A13 5G case,Galaxy A13 4G case,with HD Screen Pro... | 0.05579710144927537 | None | 9 | bm25 | yes
7 | B08KTRMH39 | 15W Wireless Charger Fast Charging Pad Slim Quick Charge ... | 0.055770149253731346 | None | 

## Cell 6 - Field Contract Check

This cell checks one HyPE unit, one proposition unit, and the joined item. It matters because field-name mismatches between indexing and search are a common integration failure. Expected output is PASS/FAIL per required field.


In [9]:
# Cell 6 — Field Contract Check
if "retrieval_units" not in dir() or "items" not in dir():
    raise RuntimeError("MongoDB chưa kết nối. Hãy chạy cell Setup trước.")

def get_path(doc, path):
    current = doc
    for part in path.split("."):
        if not isinstance(current, dict) or part not in current:
            return False, None
        current = current[part]
    return current is not None, current

def value_preview(value, max_len=80):
    if isinstance(value, list):
        if len(value) > 6:
            return f"list(len={len(value)}, first={value[:3]})"
        return str(value)
    text = "" if value is None else str(value).replace("\n", " ")
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

def print_contract_table(title, doc, fields):
    print(f"\n{title}")
    print("Field | Status | Value Preview")
    print("--- | --- | ---")
    passed = True
    for field in fields:
        exists, value = get_path(doc, field)
        if field == "embedding" and exists:
            exists = isinstance(value, list) and len(value) == 1024
        passed = passed and exists
        print(f"{field} | {'PASS' if exists else 'FAIL'} | {value_preview(value)}")
    return passed

hype_unit = retrieval_units.find_one({"unit_type": "hype_question", "embedding": {"$exists": True}})
proposition_unit = retrieval_units.find_one({"unit_type": "proposition", "text_search": {"$exists": True}})
lookup_item_id = (hype_unit or proposition_unit or {}).get("item_id")
lookup_item = items.find_one({"_id": lookup_item_id}) if lookup_item_id else None

hype_fields = [
    "item_id", "unit_type", "language", "embedding", "embedding_text", "raw_text", "aspect",
    "confidence", "category_id", "price_bucket", "in_stock", "is_cold_item",
]
proposition_fields = [
    "item_id", "unit_type", "language", "text_search", "item_title_en", "item_brand",
    "confidence", "category_id", "price_bucket", "in_stock", "is_cold_item",
]
item_fields = [
    "_id", "title_en", "brand", "price_vnd", "price_bucket", "in_stock", "image_url",
    "cold_start.is_cold_item",
]

print("Selected hype_question _id:", None if hype_unit is None else hype_unit.get("_id"))
hype_contract_pass = print_contract_table("Hype unit contract", hype_unit, hype_fields)

print("\nSelected proposition _id:", None if proposition_unit is None else proposition_unit.get("_id"))
proposition_contract_pass = print_contract_table("Proposition unit contract", proposition_unit, proposition_fields)

print("\nLookup item _id:", lookup_item_id)
item_contract_pass = print_contract_table("Item contract", lookup_item, item_fields)

field_contract_pass = hype_contract_pass and proposition_contract_pass and item_contract_pass
print("\nField contract status:", "PASS" if field_contract_pass else "FAIL")


Selected hype_question _id: 9e9fdd2d-60c0-41d7-8624-631eec596129

Hype unit contract
Field | Status | Value Preview
--- | --- | ---
item_id | PASS | B0BNK971SB
unit_type | PASS | hype_question
language | PASS | en
embedding | PASS | list(len=1024, first=[-0.018194524571299553, -0.00617508077993989, -0.03026966191828251])
embedding_text | PASS | [Category: All Electronics | Brand: GiGreen | Price bucket: 100k_300k] Fast c...
raw_text | PASS | Fast charge samsung galaxy a14 5g in 30 minutes
aspect | PASS | function
confidence | PASS | 0.95
category_id | PASS | all_electronics
price_bucket | PASS | 100k_300k
in_stock | PASS | True
is_cold_item | PASS | True

Selected proposition _id: e35ab672-0acd-4ba2-a670-fbd315e7cc60

Proposition unit contract
Field | Status | Value Preview
--- | --- | ---
item_id | PASS | B0BNK971SB
unit_type | PASS | proposition
language | PASS | en
text_search | PASS | Input voltage is AC 100V-240V at 50/60Hz.
item_title_en | PASS | USB C Samsung Fast Charging Block

## Cell 7 - Integration Summary

This cell summarizes collection health, fixture generation, pipeline results, hybrid channels, and field contract status. Expected output is clear PASS/FAIL lines and `Ready for Demo: YES` only when the full path is healthy.


In [10]:
# Cell 7 — Summary (Markdown-style)
# Define defaults if earlier cells failed
collection_health_pass = collection_health_pass if "collection_health_pass" in dir() else False
pipeline_returns_results_pass = pipeline_returns_results_pass if "pipeline_returns_results_pass" in dir() else False
hybrid_channels_pass = hybrid_channels_pass if "hybrid_channels_pass" in dir() else False
field_contract_pass = field_contract_pass if "field_contract_pass" in dir() else False
fixture = fixture if "fixture" in dir() and isinstance(fixture, dict) else {}
TEST_QUERY = TEST_QUERY if "TEST_QUERY" in dir() else "<not built>"

fixture_built_from_query = bool(fixture.get("query_embedding")) and len(fixture.get("query_embedding", [])) == 1024
ready_for_demo_ui = (
    collection_health_pass
    and fixture_built_from_query    
    and pipeline_returns_results_pass
    and hybrid_channels_pass
    and field_contract_pass
)

print("## Integration Test Summary")
print()
print(f"- Test query: {TEST_QUERY}")
print(f"- Fixture built with query_processor: {'PASS' if fixture_built_from_query else 'FAIL'}")
print(f"- Collection health: {'PASS' if collection_health_pass else 'FAIL'}")
print(f"- Pipeline returns results: {'PASS' if pipeline_returns_results_pass else 'FAIL'}")
print(f"- Hybrid channels working: {'PASS' if hybrid_channels_pass else 'FAIL'}")
print(f"- Field contract: {'PASS' if field_contract_pass else 'FAIL'}")
print(f"- Ready for Demo: {'YES' if ready_for_demo_ui else 'NO'}")


## Integration Test Summary

- Test query: Fast charge samsung galaxy a14 5g
- Fixture built with query_processor: PASS
- Collection health: PASS
- Pipeline returns results: PASS
- Hybrid channels working: PASS
- Field contract: PASS
- Ready for Demo: YES
